# Neural data analysis
This notebook simulates spike trains for several neurons using Poisson processes and computes firing rates, plots a raster, and runs PCA on spike counts.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)

n_neurons = 20
duration = 2.0
mean_rate_hz = 8.0

spike_trains = []
for i in range(n_neurons):
    rate = mean_rate_hz + rng.normal(0, 3.0)
    rate = max(rate, 0.1)
    n_spikes = rng.poisson(rate * duration)
    spikes = rng.uniform(0, duration, size=n_spikes)
    spike_trains.append(np.sort(spikes))

# Raster plot
fig, ax = plt.subplots(figsize=(8, 4))
for i, spikes in enumerate(spike_trains):
    ax.vlines(spikes, i + 0.5, i + 1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Neuron')
ax.set_title('Spike raster')

rates = np.array([len(tr) / duration for tr in spike_trains])
print('Mean firing rate: {:.2f} Hz'.format(rates.mean()))

bin_size = 0.05
bins = np.arange(0, duration + bin_size, bin_size)
counts = np.zeros((len(bins) - 1, n_neurons))
for i, spikes in enumerate(spike_trains):
    counts[:, i], _ = np.histogram(spikes, bins=bins)

pca = PCA(n_components=2)
proj = pca.fit_transform(counts)
print('Explained variance ratio:', pca.explained_variance_ratio_)

fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.scatter(proj[:, 0], proj[:, 1])
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_title('PCA of binned population activity')
plt.show()
